# Header Column Categorization
Applies rule-based categorization to `Header_Column` values from `webi_report_output_headers_columns`.

**Categories (applied in priority order — first match wins):**

| Category | Signal |
|---|---|
| Date / Period | suffix `_Date`, `_Dt`, or keyword `Year`, `Month`, `Period`, `Quarter` |
| GL / Journal | prefix `GL_` or `JE_` / `Je_` |
| Policy / Insurance | keywords `Policy`, `Claims`, `Premium`, `Underwriting`, `Carrier`, `Broker` |
| Financial / Amount | keywords `Net`, `Amount`, `Dr`, `Cr`, `Balance`, `YTD`, `Sum`, `Gross`, `Commission`, `Contributions`, `Recoveries`, `FX` |
| Currency | keyword `Currency` |
| Organisation / Entity | keywords `Organisation`, `Business_Unit`, `Company`, `Cost_Centre`, `Country`, `Legal` |
| Description / Name | suffix `_Description`, `_Name`, `Name` |
| Code / Identifier | suffix `_Code`, `_Id`, `_ID`, `_Number`, `_Num`, `_Reference`, `_Type`, `_Source`, `_Category` |
| Other | everything else (empty, numeric-only, unmatched) |

In [0]:
%sql

select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables

In [0]:
from pyspark.sql.functions import col, when, trim, upper, countDistinct, count, collect_set

SOURCE_TABLE = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers_columns"
TARGET_TABLE = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers_columns_categorized"

df = spark.table(SOURCE_TABLE)

# Rule-based categorization — first matching rule wins
# Patterns are case-insensitive ((?i) flag)
def categorize(c):
    return (
        # --- Date / Period ---
        when(c.rlike(r'(?i)(^|_)(date|dt)($|_)'),              "Date / Period")
        .when(c.rlike(r'(?i)(year|month|period|quarter)'),      "Date / Period")

        # --- GL / Journal ---
        .when(c.rlike(r'(?i)^(gl_|je_|je_)'),                  "GL / Journal")

        # --- Policy / Insurance ---
        .when(c.rlike(r'(?i)(policy|claims|premium|underwriting|carrier|broker|reinsur)'), "Policy / Insurance")

        # --- Financial / Amount ---
        .when(c.rlike(r'(?i)(_net$|_net_|_amount|accounted|entered|balance|_ytd|_dr$|_cr$|gross|commission|contribution|recoveries|local_fx|^sum$|^eur$)'), "Financial / Amount")

        # --- Currency ---
        .when(c.rlike(r'(?i)currency'),                         "Currency")

        # --- Organisation / Entity ---
        .when(c.rlike(r'(?i)(organisation|business_unit|company|cost_centre|legal_|unit_org|country(?!_code))'), "Organisation / Entity")

        # --- Description / Name ---
        .when(c.rlike(r'(?i)(_description$|_name$|^name$)'),   "Description / Name")

        # --- Code / Identifier ---
        .when(c.rlike(r'(?i)(_code$|_id$|_id_|_number$|_num$|_reference|_type$|_source$|_category$|_set$)'), "Code / Identifier")

        # --- Other ---
        .otherwise("Other")
    )

df_categorized = df.withColumn("Category", categorize(col("Header_Column")))

print(f"Total rows: {df_categorized.count():,}")
display(df_categorized)

In [0]:


total = df_categorized.count()

dist = (
    df_categorized
    .groupBy("Category")
    .agg(
        count("*").alias("Column_Count"),
        countDistinct("Report_Document_ID").alias("Report_Count")
    )
    .orderBy(col("Column_Count").desc())
)

print(f"Category distribution across {total:,} rows:")
display(dist)
category_columns = (
    df_categorized
    .select("Category", "Header_Column")
    .orderBy("Category", "Header_Column")
).distinct()

display(category_columns)

In [0]:
# Inspect what lands in 'Other' to refine rules if needed
df_other = (
    df_categorized
    .filter(col("Category") == "Other")
    .groupBy("Header_Column")
    .agg(count("*").alias("freq"))
    .orderBy(col("freq").desc())
)

print("Top uncategorized Header_Column values:")
display(df_other)

In [0]:
(
    df_categorized
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)
print(f"Written to {TARGET_TABLE}")
display(spark.table(TARGET_TABLE).limit(20))